# Practice Lab: Math Foundations 4 - Transformer Plumbing

8 exercises on LayerNorm, RMSNorm, positional encoding, residuals.

In [ ]:
!pip install -q numpy torch matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

## Exercise 1 - LayerNorm by hand

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
mean = x.mean(dim=-1, keepdim=True)
var = x.var(dim=-1, keepdim=True, unbiased=False)
my_ln = (x - mean) / torch.sqrt(var + 1e-5)
print(f'by hand: {my_ln.round(decimals=3)}')

ln = nn.LayerNorm(4)
print(f'pytorch: {ln(x).round(decimals=3)}')

## Exercise 2 - RMSNorm vs LayerNorm

In [ ]:
x = torch.tensor([[3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0]])
ln = nn.LayerNorm(8)
rms = nn.RMSNorm(8)
ln_out = ln(x); rms_out = rms(x)
print(f'LayerNorm: mean={ln_out.mean().item():.3f}, std={ln_out.std(unbiased=False).item():.3f}')
print(f'RMSNorm:   mean={rms_out.mean().item():.3f}, RMS={(rms_out**2).mean().sqrt().item():.3f}')

## Exercise 3 - Sinusoidal PE heatmap

In [ ]:
def sinusoidal_pe(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(16, 32)
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.xlabel('dimension'); plt.ylabel('position'); plt.colorbar(); plt.title('Sinusoidal PE')
plt.show()
print(f'PE shape: {pe.shape}')

## Exercise 4 - Vanishing gradient demo

In [ ]:
class Block(nn.Module):
    def __init__(self, d, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.layer = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        f = self.layer(x)
        return x + f if self.use_residual else f

def grad_norm(use_residual, depth=20):
    torch.manual_seed(0)
    blocks = nn.Sequential(*[Block(8, use_residual) for _ in range(depth)])
    x = torch.randn(1, 8, requires_grad=True)
    y = blocks(x).sum()
    y.backward()
    return x.grad.norm().item()

print(f'20 layers, NO residual:   grad norm = {grad_norm(False):.3e}')
print(f'20 layers, WITH residual: grad norm = {grad_norm(True):.3e}')

## Exercise 5 - Pre-LN vs Post-LN convergence

In [ ]:
class Block2(nn.Module):
    def __init__(self, d, pre_ln=True):
        super().__init__()
        self.pre_ln = pre_ln
        self.norm1 = nn.LayerNorm(d); self.norm2 = nn.LayerNorm(d)
        self.ffn1 = nn.Linear(d, d); self.ffn2 = nn.Linear(d, d)
    def forward(self, x):
        if self.pre_ln:
            x = x + F.gelu(self.ffn1(self.norm1(x)))
            x = x + self.ffn2(self.norm2(x))
        else:
            x = self.norm1(x + F.gelu(self.ffn1(x)))
            x = self.norm2(x + self.ffn2(x))
        return x

def train(pre_ln, n_steps=200):
    torch.manual_seed(0)
    model = nn.Sequential(*[Block2(16, pre_ln) for _ in range(12)])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randn(32, 16); y = torch.randn(32, 16)
    losses = []
    for _ in range(n_steps):
        opt.zero_grad()
        loss = F.mse_loss(model(x), y)
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

pre = train(True); post = train(False)
print(f'Pre-LN  final: {pre[-1]:.4f}')
print(f'Post-LN final: {post[-1]:.4f}')

## Exercise 6 - RoPE rotation

In [ ]:
def rope_rotate(Q, base=10000):
    seq_len, d_head = Q.shape
    positions = torch.arange(seq_len).float().unsqueeze(1)
    freqs = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    angles = positions * freqs
    cos = torch.cos(angles); sin = torch.sin(angles)
    Q_even = Q[:, 0::2]; Q_odd = Q[:, 1::2]
    rotated_even = Q_even * cos - Q_odd * sin
    rotated_odd = Q_even * sin + Q_odd * cos
    out = torch.zeros_like(Q)
    out[:, 0::2] = rotated_even
    out[:, 1::2] = rotated_odd
    return out

torch.manual_seed(0)
Q = torch.randn(4, 8)
Q_rot = rope_rotate(Q)
print(f'original norms per row: {Q.norm(dim=-1).round(decimals=3).tolist()}')
print(f'rotated norms per row:  {Q_rot.norm(dim=-1).round(decimals=3).tolist()}')
assert torch.allclose(Q.norm(dim=-1), Q_rot.norm(dim=-1), atol=1e-5)

## Exercise 7 - Full Pre-LN block from scratch

In [ ]:
class PreLNBlock(nn.Module):
    def __init__(self, d=64, n_heads=4):
        super().__init__()
        self.norm1 = nn.RMSNorm(d); self.norm2 = nn.RMSNorm(d)
        self.qkv = nn.Linear(d, 3*d)
        self.out_proj = nn.Linear(d, d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.n_heads = n_heads; self.d_head = d // n_heads
    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        a = a.transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.out_proj(a)
        x = x + self.ffn(self.norm2(x))
        return x

torch.manual_seed(0)
block = PreLNBlock()
x = torch.randn(2, 8, 64)
out = block(x)
print(f'output shape: {out.shape}')

## Exercise 8 - YaRN-style RoPE base scaling

In [ ]:
def rope_rotate_scaled(Q, base=10000, scale=1.0):
    seq_len, d_head = Q.shape
    positions = torch.arange(seq_len).float().unsqueeze(1)
    effective_base = base * scale
    freqs = 1.0 / (effective_base ** (torch.arange(0, d_head, 2).float() / d_head))
    angles = positions * freqs
    cos = torch.cos(angles); sin = torch.sin(angles)
    Q_even = Q[:, 0::2]; Q_odd = Q[:, 1::2]
    rotated_even = Q_even * cos - Q_odd * sin
    rotated_odd = Q_even * sin + Q_odd * cos
    out = torch.zeros_like(Q)
    out[:, 0::2] = rotated_even
    out[:, 1::2] = rotated_odd
    return out

torch.manual_seed(0)
def attn_score_var(seq_len, base=10000, scale=1.0):
    Q = torch.randn(seq_len, 16)
    K = torch.randn(seq_len, 16)
    Q_rope = rope_rotate_scaled(Q, base, scale)
    K_rope = rope_rotate_scaled(K, base, scale)
    return (Q_rope @ K_rope.T).var().item()

print(f'Seq 32,  scale=1.0 -> attn var: {attn_score_var(32):.2f}')
print(f'Seq 128, scale=1.0 -> attn var: {attn_score_var(128):.2f}')
print(f'Seq 128, scale=4.0 -> attn var: {attn_score_var(128, scale=4.0):.2f}')

## Wrap-up

Exercise 4 (vanishing gradient) is the most memorable demo. Exercise 7 (Pre-LN block) is the most useful Module 4 prep.